# Leistungsverzeichnis (LV) Extraction & Structured Data Table

This notebook extracts and parses German Construction Bill of Quantities (*Leistungsverzeichnis* / LV) from PDF files into a structured tabular `pandas.DataFrame`.

The final table includes:
- **`oz`**: Ordnungszahl (position number)
- **`position_type`**: Position classification (`Normalposition`, `Wahlposition`, `Grundposition`, `Bedarfsposition`)
- **`short_text`**: Short title / summary of the work item
- **`quantity`**: Numerical quantity
- **`unit`**: Unit of measurement (e.g. `m²`, `m`, `St`, `h`)
- **`only_unit_price`**: Boolean indicating whether only a unit price applies (`Nur Einh.-Pr.`)
- **`long_text`**: Detailed description / specifications


In [1]:
import os
import re
import pdfplumber
import pandas as pd

# Set display options for clear table visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 1000)


In [2]:
# Regex patterns tailored to this LV format
OZ_PATTERN = re.compile(r"(?m)^(\.\d+)\s+")
QTY_UNIT_PATTERN = re.compile(
    r"(\d{1,3}(?:\.\d{3})*,\d{3})\s+(m²|m|St|Set|Psch|psch|Satz|h)"
)
NOISE_PATTERNS = [
    r"Übertrag:.*",
    r"Druckdatum:.*",
    r"Seite:\s*\d+",
    r"Summe LV.*",
]


def extract_clean_text(pdf_path: str, top_margin: int = 120, bottom_margin: int = 750) -> str:
    """Extracts text while cropping out repeating headers and footers."""
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            bbox = (0, top_margin, page.width, min(bottom_margin, page.height))
            cropped = page.crop(bbox)
            text = cropped.extract_text(layout=False) or ""
            pages_text.append(text)

    full_text = "\n".join(pages_text)
    for pattern in NOISE_PATTERNS:
        full_text = re.sub(pattern, "", full_text)
    return full_text


def parse_german_float(num_str: str) -> float:
    """Converts '3.300,000' -> 3300.0"""
    return float(num_str.replace(".", "").replace(",", "."))


def parse_lv_positions(full_text: str) -> pd.DataFrame:
    """Splits continuous LV text into structured rows per OZ."""
    matches = list(OZ_PATTERN.finditer(full_text))
    records = []

    for i, match in enumerate(matches):
        oz = match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        block = full_text[start:end].strip()

        # Detect position type
        pos_type = "Normalposition"
        if "Wahlposition" in block:
            pos_type = "Wahlposition"
        elif "Bedarfsposition" in block:
            pos_type = "Bedarfsposition"
        elif "Grundposition" in block:
            pos_type = "Grundposition"

        only_unit_price = "Nur Einh.-Pr." in block

        qty_match = QTY_UNIT_PATTERN.search(block)
        quantity = parse_german_float(qty_match.group(1)) if qty_match else None
        unit = qty_match.group(2) if qty_match else None

        lines = [line.strip() for line in block.splitlines() if line.strip()]
        title_idx = 1 if lines and ("position" in lines[0].lower()) else 0
        short_text = lines[title_idx] if len(lines) > title_idx else ""
        long_text = "\n".join(lines[title_idx + 1 :])

        records.append({
            "oz": oz,
            "position_type": pos_type,
            "short_text": short_text,
            "quantity": quantity,
            "unit": unit,
            "only_unit_price": only_unit_price,
            "long_text": long_text,
        })

    return pd.DataFrame(records)


## 1. Extract and Parse the PDF

In [3]:
# Locate target PDF
pdf_filename = "LV_14a_Betonkernaktivierung.pdf"
candidates = [
    pdf_filename,
    os.path.join("..", "Data", pdf_filename),
    os.path.join("Data", pdf_filename),
]
pdf_path = next((c for c in candidates if os.path.exists(c)), None)

if not pdf_path:
    raise FileNotFoundError(f"Could not find {pdf_filename}")

print(f"Loading PDF from: {pdf_path}")
raw_text = extract_clean_text(pdf_path)
print(f"Total raw characters extracted: {len(raw_text):,}")

df = parse_lv_positions(raw_text)
print(f"Parsed {len(df)} positions successfully.")


Loading PDF from: ..\Data\LV_14a_Betonkernaktivierung.pdf


Total raw characters extracted: 25,562
Parsed 36 positions successfully.


## 2. The Final Parsed Table

This is the main structured dataset produced by the parser:


In [4]:
# Display the complete table without long_text clutter
summary_columns = ["oz", "position_type", "short_text", "quantity", "unit", "only_unit_price"]
df[summary_columns]


,oz,position_type,short_text,quantity,unit,only_unit_price
0,.1,Grundposition,Rohrregister zur Betonkernaktivierung in Modulbauweise,None,None,False
1,.2,Wahlposition,Alternativ : Handverlegtes PE- Xa - Rohr mit allem Zubehör,None,None,False
2,.3,Normalposition,Anbindeleitung ( für die Bauteilaktivierung),None,None,False
3,.4,Normalposition,Anbindeleitung ( für die Thermischen Steckdosen ),None,None,False
4,.5,Normalposition,Schutzrohr für Anbindeleitung,None,None,False
5,.6,Normalposition,"Thermische Steckdose für PE- Xa Rohr 2 0 x2 ,3",None,None,False
6,.7,Normalposition,Adapterstecker für Thermische Steckdose,None,None,False
7,.8,Normalposition,Uponor Press- Kupplung PE - Xa 2 0-20 mit 2 Presshülsen,None,None,False
8,.9,Normalposition,"Heiz- u nd Kühlkreisverteiler, 8 Kreise",None,None,False
9,.1,Normalposition,"0 Heiz- u nd Kühlkreisverteiler, 7 Kreise",None,None,False


## 3. Position Types & Quantities Breakdown

In [5]:
# Summary statistics
print("=== Position Types Count ===")
print(df["position_type"].value_counts())

print("\n=== Units Count ===")
print(df["unit"].value_counts(dropna=False))

print("\n=== Total Quantities by Unit ===")
print(df.groupby("unit")["quantity"].agg(["count", "sum", "min", "max"]))


=== Position Types Count ===
position_type
Normalposition     28
Bedarfsposition     6
Grundposition       1
Wahlposition        1
Name: count, dtype: int64

=== Units Count ===
unit
None    36
Name: count, dtype: int64

=== Total Quantities by Unit ===
Empty DataFrame
Columns: [count, sum, min, max]
Index: []


## 4. Inspecting Full Position Details (Long Text)

In [6]:
# Preview first item with long specification description
first_row = df.iloc[0]
print(f"OZ:            {first_row['oz']}")
print(f"Type:          {first_row['position_type']}")
print(f"Short text:    {first_row['short_text']}")
print(f"Quantity:      {first_row['quantity']} {first_row['unit']}")
print(f"Unit price:    {first_row['only_unit_price']}")
print(f"\n--- Long Text Description ---\n{first_row['long_text'][:400]}...")


OZ:            .1
Type:          Grundposition
Short text:    Rohrregister zur Betonkernaktivierung in Modulbauweise
Quantity:      None None
Unit price:    False

--- Long Text Description ---
( Einbausituation Betondecke)
Rohrregister in Form von vorgefertigten Modulen mit
integrierten Anbindeleitungen individueller Länge zur
Montage zwischen bauseitiger unterer Bewehrung und
oberer Bewehrung zur thermischen Aktivierung der
OZ L eistungsbeschreibung M enge ME E inheitspreis G esamtbetrag
i n EUR in EUR
Betondecke, b estehend aus :
- w erkseitig objektbezogen komplett vorgefertigtem
Mod...


## 5. Export Structured Data

In [7]:
# Export to CSV and JSON
csv_path = "lv_structured.csv"
json_path = "lv_structured.json"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"Exported {len(df)} rows:")
print(f"  - CSV:  {csv_path} ({os.path.getsize(csv_path):,} bytes)")
print(f"  - JSON: {json_path} ({os.path.getsize(json_path):,} bytes)")


Exported 36 rows:
  - CSV:  lv_structured.csv (22,830 bytes)
  - JSON: lv_structured.json (28,229 bytes)
